# 🧠 Hybrid RoBERTa + Tabular Classifier with Batch Inference
This notebook builds a binary classifier for customer reviews using:
- `review_text` → RoBERTa embeddings
- `rating`, `delivery_delay`, `is_verified` → Tabular features
- Labels: `positive`, `positive_irrelevant`

It includes training and batch inference with confidence + uncertainty flags.

In [ ]:
# 📦 Install libraries!pip install -q transformers scikit-learn

In [ ]:
# ✅ Importsimport pandas as pdimport numpy as npimport torchfrom sklearn.preprocessing import LabelEncoder, StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_reportfrom transformers import RobertaTokenizer, RobertaModelimport torch.nn as nnimport torch.optim as optim

In [ ]:
# 🧪 Simulated + Extended Datasettexts = [    "Great product, works as expected.",    "Looks nice, haven't tried it yet.",    "Very satisfied with the performance.",    "Package arrived on time, not used yet.",    "Loved it! Highly recommended.",    "Nice packaging, still evaluating.",    "Absolutely wonderful. Will buy again.",    "Seems fine, not tested yet.",    "Excellent quality and easy to use.",    "Got it yesterday, looks okay so far.",    "Excellent quality, used it every day for a week.",    "Loved it! Exactly what I needed.",    "Great value for money. Very satisfied.",    "Amazing performance after using it thoroughly.",    "Highly recommended, this has changed my workflow.",    "Arrived on time, haven't tested it yet.",    "Looks nice in the box, not used.",    "Can't comment on functionality yet.",    "Still evaluating. So far looks fine.",    "Just received it, will update later."]labels = [    "positive", "positive_irrelevant", "positive", "positive_irrelevant", "positive",    "positive_irrelevant", "positive", "positive_irrelevant", "positive", "positive_irrelevant",    "positive", "positive", "positive", "positive", "positive",    "positive_irrelevant", "positive_irrelevant", "positive_irrelevant", "positive_irrelevant", "positive_irrelevant"]df = pd.DataFrame({    'review_text': texts,    'rating': np.random.randint(3, 6, size=20),    'delivery_delay': np.random.randint(0, 5, size=20),    'is_verified': np.random.choice([0, 1], size=20),    'label': labels})

In [ ]:
# 🔤 Encode labels and normalize featureslabel_encoder = LabelEncoder()df["label_id"] = label_encoder.fit_transform(df["label"])scaler = StandardScaler()tabular = scaler.fit_transform(df[["rating", "delivery_delay", "is_verified"]])

In [ ]:
# 🔎 Tokenize and embed with RoBERTatokenizer = RobertaTokenizer.from_pretrained("roberta-base")tokens = tokenizer(list(df["review_text"]), padding=True, truncation=True, return_tensors='pt', max_length=128)model_roberta = RobertaModel.from_pretrained("roberta-base")with torch.no_grad():    cls_embeddings = model_roberta(**tokens).last_hidden_state[:, 0, :].numpy()

In [ ]:
# 🔀 Combine features and splitX = np.concatenate([cls_embeddings, tabular], axis=1)y = df["label_id"].valuesX_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

In [ ]:
# 🤖 Classifierclass Classifier(nn.Module):    def __init__(self, in_dim):        super().__init__()        self.net = nn.Sequential(            nn.Linear(in_dim, 128),            nn.ReLU(),            nn.Linear(128, 2)        )    def forward(self, x):        return self.net(x)model = Classifier(X.shape[1])opt = optim.Adam(model.parameters(), lr=1e-3)loss_fn = nn.CrossEntropyLoss()

In [ ]:
# 🏋️ TrainX_train_tensor = torch.tensor(X_train, dtype=torch.float32)y_train_tensor = torch.tensor(y_train, dtype=torch.long)for epoch in range(15):    opt.zero_grad()    pred = model(X_train_tensor)    loss = loss_fn(pred, y_train_tensor)    loss.backward()    opt.step()    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

In [ ]:
# 📊 EvaluateX_test_tensor = torch.tensor(X_test, dtype=torch.float32)with torch.no_grad():    logits = model(X_test_tensor)    y_pred = torch.argmax(logits, dim=1).numpy()print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# 🔍 Inference function with uncertaintydef predict_review(text, rating, delivery_delay, is_verified):    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)    with torch.no_grad():        roberta_output = model_roberta(**inputs)        cls_embed = roberta_output.last_hidden_state[:, 0, :].numpy()    tabular_input = scaler.transform([[rating, delivery_delay, is_verified]])    combined_input = np.concatenate([cls_embed, tabular_input], axis=1)    combined_tensor = torch.tensor(combined_input, dtype=torch.float32)    with torch.no_grad():        logits = model(combined_tensor)        probs = torch.softmax(logits, dim=1).numpy()[0]        pred_class = np.argmax(probs)        label = label_encoder.inverse_transform([pred_class])[0]        uncertainty_flag = abs(probs[0] - probs[1]) < 0.1    return label, probs, uncertainty_flag

In [ ]:
# 🧪 Run batch inferencetest_cases = [    ("Great performance after several weeks of use.", 5, 1, 1),    ("Looks amazing, but haven't tested it.", 4, 2, 0),    ("Highly recommended after using extensively.", 5, 0, 1),    ("Nice packaging, still evaluating.", 4, 3, 0),    ("Absolutely loved it! Works as expected.", 5, 1, 1),    ("Got it quickly, haven't used it yet.", 5, 1, 1),]for text, rating, delay, verified in test_cases:    label, probs, uncertain = predict_review(text, rating, delay, verified)    print(f"Review: {text}")    print(f"=> Predicted: {label} | Confidence: {probs} | Uncertain? {'⚠️ Yes' if uncertain else 'No'}\n")